# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_09 — AI_SIMILARITY_PANDAS
## 1. Objetivos y requisitos
Comparar textos por significado contra una referencia fija y por pares, validar puntuaciones, ordenar resultados y auditar el consumo. Requiere Fabric, AI Functions habilitadas y runtime Python compatible. Datos ficticios; las inferencias consumen capacidad.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 3, Finished, Available, Finished, False)

## 2. Configuración
Modelo de embeddings Ada, no modelo generativo. Concurrencia de laboratorio 10 y barra de progreso con estadísticas. Usar una sesión limpia con endpoint integrado.

In [2]:
import pandas as pd
import numpy as np
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "text-embedding-ada-002"
CONCURRENCY = 10
aifunc.default_conf.embedding_deployment_name = MODEL_NAME
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"
RUN_PAIRWISE = True
THRESHOLD = 0.80

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.similarity con Pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/similarity)

```python
df["similarity"] = df["col1"].ai.similarity("value")
df["similarity"] = df["col1"].ai.similarity(df["col2"])
```

| Parámetro | Tipo | Requisito |
|---|---|---|
| `other` | str o pandas.Series | Obligatorio; referencia común o serie de igual dimensión. |

Devuelve una Series de puntuaciones entre -1 y 1, útiles para ranking. Una contradicción puede tener puntuación alta por compartir vocabulario y tema. No es una probabilidad. Los ejercicios siguientes son propios.

## 4. Dataset ficticio
Se conservan los seis pares del NB_09 PySpark, incluida la contradicción del ticket 5.

In [3]:
data = [
    (1, "No puedo acceder al portal corporativo", "No puedo acceder al portal corporativo"),
    (2, "El inicio de sesión falla", "No consigo entrar al sistema"),
    (3, "El pipeline de ventas falló", "La carga nocturna de ventas no terminó"),
    (4, "Necesito una silla ergonómica", "Restablecer contraseña"),
    (5, "El portal funciona correctamente", "El portal no funciona correctamente"),
    (6, "Olvidé mi contraseña", "Necesito recuperar mis credenciales")
]
df = pd.DataFrame(data, columns=["id_ticket", "descripcion", "texto_comparacion"])
REFERENCIA = "Problemas de acceso al portal corporativo"
display(df)

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5bf9ca3e-efac-41be-8973-732564fe54b0)

## 5. Comparación contra referencia fija
Capturar las estadísticas inmediatamente después de cada llamada y antes de transformar la Series. Cada modalidad se audita por separado.

In [4]:
runs = []
def ejecutar(mode, other):
    rid = str(uuid.uuid4())
    started = datetime.now(timezone.utc).isoformat()
    cols = ["descripcion", "texto_comparacion"] if isinstance(other, pd.Series) else ["descripcion"]
    input_bytes = int(df[cols].memory_usage(deep=True).sum())
    if isinstance(other, str):
        import sys
        input_bytes += sys.getsizeof(other)
    t0 = perf_counter()
    scores = df["descripcion"].ai.similarity(other)
    elapsed = perf_counter() - t0
    raw = scores.ai.stats
    runs.append(dict(run_id=rid, started=started, mode=mode, scores=scores, raw=raw, elapsed=elapsed, input_bytes=input_bytes))
    return scores
scores_fixed = ejecutar("referencia_fija", REFERENCIA)
resultado = df.assign(similarity=scores_fixed)
display(resultado)

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 6, Finished, Available, Finished, False)

ai.similarity: 100%|██████████| 6/6 [00:00<00:00, 14.54it/s, cached=0, in=49, out=0, CU-s=0.16]


SynapseWidget(Synapse.DataFrame, 4925c017-4848-45dc-8833-47cb34df14e1)

## 6. Comparación por pares
La segunda llamada consume tokens adicionales. Cambiar RUN_PAIRWISE a False para omitirla. Las series provienen del mismo DataFrame y conservan su índice.

In [5]:
resultado_pares = None
if RUN_PAIRWISE:
    scores_pairs = ejecutar("pares", df["texto_comparacion"])
    resultado_pares = df.assign(similarity=scores_pairs)
    display(resultado_pares)

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 7, Finished, Available, Finished, False)

ai.similarity: 100%|██████████| 6/6 [00:00<00:00, 13.78it/s, cached=0, in=83, out=0, CU-s=0.28]                         


SynapseWidget(Synapse.DataFrame, c2e61ccf-6591-4c86-8fa8-308b1789e1bc)

## 7. Validación y ranking
El umbral 0.80 es ilustrativo, no una frontera universal. Los resultados ausentes no identifican por sí mismos la causa del fallo. Comparar los dos rankings y revisar manualmente la contradicción del ticket 5.

In [6]:
def validar(score):
    if pd.isna(score):
        return "MISSING"
    if not isinstance(score, (int, float, np.integer, np.floating)) or not np.isfinite(score) or not -1.000001 <= score <= 1.000001:
        return "INVALID"
    return "OK"
for run in runs:
    frame = df.assign(similarity=run["scores"])
    frame["validation_status"] = frame.similarity.map(validar)
    run["valid"] = frame.validation_status.eq("OK").all()
    print(run["mode"])
    display(frame.sort_values("similarity", ascending=False, na_position="last"))
    display(frame.validation_status.value_counts(dropna=False))
    display(frame.loc[frame.validation_status.eq("OK") & frame.similarity.ge(THRESHOLD)])

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 8, Finished, Available, Finished, True)

referencia_fija


SynapseWidget(Synapse.DataFrame, 7a3e487b-86bd-4e38-9648-5fe38ab8e928)

SynapseWidget(Synapse.DataFrame, 07a29d6d-0641-4395-aa35-7a1ec05ea2d7)

SynapseWidget(Synapse.DataFrame, 16e7c57b-e4a9-4c1d-a8b5-c4ae73b1a9de)

pares


SynapseWidget(Synapse.DataFrame, 82e82ec9-7b63-4f0f-bca2-bde0757edb18)

SynapseWidget(Synapse.DataFrame, 7d7c5828-009a-465a-a648-df101478f480)

SynapseWidget(Synapse.DataFrame, 7910f0d4-00b0-497d-ac41-3d925df81674)

## 8. Estadísticas y consumo de capacidad
[Facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing): Ada consume 3,36 CU-s por 1000 tokens de entrada. No se multiplican los tokens nuevamente por el número de pares. No incluye cómputo del notebook ni almacenamiento. Solo estimar si el modelo y endpoint reportados coinciden.

$$CU_s=I\times3.36/1000$$

Ejemplo hipotético: 1000 tokens = 3,36 CU-s.

In [7]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")


def metric(stats, k):
    v = stats.get(k)
    return None if v is None or pd.isna(v) else int(v)
for run in runs:
    display(run["raw"])
    stats = normalizar(run["raw"])
    run["stats"] = stats
    I = metric(stats, "input_tokens")
    run["cu"] = I * 3.36 / 1000 if I is not None and I >= 0 and stats.get("model") == MODEL_NAME and stats.get("client_type") == "fabric_llm_endpoint" else None
    print(run["mode"], "tokens:", I, "CU-s:", run["cu"])

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 9, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 7cc07bae-74fd-462b-9462-91b14c7a0cb8)

referencia_fija tokens: 49 CU-s: 0.16463999999999998


SynapseWidget(Synapse.DataFrame, 2e02c696-b388-4bda-862c-8194dfcc4d18)

pares tokens: 83 CU-s: 0.27888


## 9. Auditoría individual y acumulada
Esquema compatible con NB_08 Pandas. La modalidad se conserva en una vista adicional. La memoria es aproximada, no el pico del proceso. Si falta una estimación no se presenta un total engañoso.

In [8]:
records = []
for run in runs:
    stats = run["stats"]
    audit = dict(run_id=run["run_id"], execution_timestamp_utc=run["started"], notebook_name="NB_09_AI_SIMILARITY_PANDAS", operation="ai.similarity", model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=len(df), client_type=stats.get("client_type"), estimated_cu_seconds=run["cu"], elapsed_seconds=run["elapsed"], input_bytes=run["input_bytes"], output_bytes=int(run["scores"].memory_usage(deep=True)), concurrency=CONCURRENCY)
    for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
        audit[key] = metric(stats, key)
    counts = [audit[k] for k in ("num_successful","num_exceptions","num_unevaluated","num_harmful")]
    if any(v is None for v in counts):
        status = "UNKNOWN"
    elif counts[1] or counts[3]:
        status = "PARTIAL"
    elif counts == [len(df),0,0,0] and run["valid"]:
        status = "SUCCESS"
    else:
        status = "INCOMPLETE"
    audit["execution_status"] = status
    records.append(audit)
audit_df = pd.DataFrame(records)
display(audit_df.assign(modalidad=[r["mode"] for r in runs]))
print("CU-s acumulados:", sum(r["cu"] for r in runs) if all(r["cu"] is not None for r in runs) else None)

StatementMeta(, 30e9a973-a409-490e-bcea-468e2debeaa6, 10, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 5259fa8a-1be8-4c52-81e4-835a3c6b3163)

CU-s acumulados: 0.44352


## 10. Persistencia opcional de auditoría
Requiere Lakehouse adjunto, permisos y paquetes deltalake y pyarrow. Ruta bajo Files, no tabla SQL registrada. Reejecutar la escritura agrega registros duplicados.

In [ ]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

## 11. Interpretación
Una puntuación alta no implica equivalencia factual. Para detectar duplicados, calibrar con ejemplos etiquetados. Para búsqueda a escala, considerar vectores persistidos y reutilización de embeddings (NB_08). No repetir llamadas solo para recalcular estadísticas.

## 12. Comparación con PySpark
| Pandas | PySpark |
|---|---|
| `df["descripcion"].ai.similarity(REFERENCIA)` | `df.ai.similarity(input_col="descripcion", other=REFERENCIA)` |
| `df["descripcion"].ai.similarity(df["texto_comparacion"])` | `df.ai.similarity(input_col="descripcion", other_col="texto_comparacion")` |
| Series local, conservar ai.stats | DataFrame distribuido, materializar y conservar resultados |

## 13. Conclusiones y referencias
Las dos modalidades permiten comparar significado; contrastar las estimaciones con Capacity Metrics, operación AI Functions bajo Copilot and AI. Validación estática, sin ejecución en Fabric.
- [Método Pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/similarity)
- [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

Fin de la serie Pandas.